# Chapter 15 Part III Capstone Coding Exam — Reproducible Research Engineering (100 Points)

**Book Reference:** Chapter 15, Section 15.7 (Part III Capstone Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | Reproducible Engineering Function | Points |
| :--- | :--- | :---: |
| **Task 15.1** | `audit_repo_hygiene(repo_dir)` (Pre-push repository hygiene gatekeeper) | 30 |
| **Task 15.2** | `verify_data_provenance(data_dir, manifest_dict)` (SHA-256 dataset auditor) | 35 |
| **Task 15.3** | `run_logged_cleaning_pipeline(in_csv, out_json, log_path)` (End-to-end logged pipeline) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q sharukh-tutor
import sharukh_tutor as tutor

tutor.activate(chapter="ch15", notebook="coding_exam")


---
## Exam Task 15.1 (30 Points): Pre-Push Repository Hygiene Auditor

Write `audit_repo_hygiene(repo_dir)` that inspects `repo_dir` (`Path` or `str`) and **returns** a dictionary containing five `bool` flags:
1. `"has_readme"`: `True` if `README.md` exists as a regular file inside `repo_dir`.
2. `"has_gitignore"`: `True` if `.gitignore` exists as a regular file inside `repo_dir`.
3. `"has_env_leak"`: `True` if a `.env` file exists inside `repo_dir`.
4. `"has_checkpoints"`: `True` if a `.ipynb_checkpoints` directory exists inside `repo_dir`.
5. `"ready_to_push"`: `has_readme and has_gitignore and not has_env_leak and not has_checkpoints`.

In [ ]:
from pathlib import Path

def audit_repo_hygiene(repo_dir):
    """Inspect repo_dir for docs, .gitignore, .env leaks & checkpoints."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    root = Path(repo_dir)
    has_readme = (root / "README.md").is_file()
    has_gitignore = (root / ".gitignore").is_file()
    has_env_leak = (root / ".env").is_file()
    has_checkpoints = (root / ".ipynb_checkpoints").is_dir()
    ready_to_push = (
        has_readme
        and has_gitignore
        and not has_env_leak
        and not has_checkpoints
    )
    return {
        "has_readme": has_readme,
        "has_gitignore": has_gitignore,
        "has_env_leak": has_env_leak,
        "has_checkpoints": has_checkpoints,
        "ready_to_push": ready_to_push,
    }
    ### END SOLUTION


---
## Exam Task 15.2 (35 Points): Cryptographic Dataset Provenance Verifier

Write `verify_data_provenance(data_dir, manifest_dict)` where `manifest_dict` maps each `filename` to `{"bytes": int, "sha256": str}` (matching `build_dataset_manifest`):
1. Check every filename in `sorted(manifest_dict)` inside `data_dir`.
2. If the file does not exist on disk, add its name to `missing`.
3. Otherwise recompute its SHA-256 hex digest: add to `matched` if equal to `manifest_dict[fname]["sha256"]`, else add to `modified`.
4. **Return** `{"matched": sorted_list, "modified": sorted_list, "missing": sorted_list, "all_verified": bool}` where `all_verified` is `True` iff both `modified` and `missing` are empty.

In [ ]:
import hashlib
from pathlib import Path

def verify_data_provenance(data_dir, manifest_dict):
    """Verify data_dir files against manifest_dict SHA-256 digests."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    root = Path(data_dir)
    matched = []
    modified = []
    missing = []
    for fname in sorted(manifest_dict):
        fpath = root / fname
        if not fpath.is_file():
            missing.append(fname)
            continue
        actual_sha = hashlib.sha256(fpath.read_bytes()).hexdigest()
        if actual_sha == manifest_dict[fname]["sha256"]:
            matched.append(fname)
        else:
            modified.append(fname)
    return {
        "matched": matched,
        "modified": modified,
        "missing": missing,
        "all_verified": len(modified) == 0 and len(missing) == 0,
    }
    ### END SOLUTION


---
## Exam Task 15.3 (35 Points): Logged, Provenance-Tracked Cleaning Pipeline

Write `run_logged_cleaning_pipeline(in_csv, out_json, log_path)` to tie together CSV I/O, SHA-256 provenance, structured logging, and JSON persistence:
1. Compute the lowercase 64-char SHA-256 hex digest of `in_csv`.
2. Attach a UTF-8 `logging.FileHandler` to `log_path` (always removing and closing it in a `finally:` block!) and emit `INFO` start/completion lines.
3. Stream `in_csv` via `csv.DictReader` expecting column `hb_g_dl`: skip `""`, `"N/A"`, and `"999"` while counting `rejected_count` and logging a `WARNING` for each rejected row; collect valid `float` values.
4. Compute `mean_hb = round(sum(valid) / len(valid), 2)` if valid rows exist, else `None`.
5. Write indented UTF-8 JSON (`indent=2`) with keys `{"source_sha256", "valid_count", "rejected_count", "mean_hb"}` to `out_json` and **return** that dictionary.

In [ ]:
import csv
import hashlib
import json
import logging
from pathlib import Path

def run_logged_cleaning_pipeline(in_csv, out_json, log_path):
    """Hash, clean, log, and summarize hb_g_dl CSV to indented JSON."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    src_sha = hashlib.sha256(Path(in_csv).read_bytes()).hexdigest()
    logger = logging.getLogger("ch15_capstone_pipeline")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)

    valid_vals = []
    rejected = 0
    try:
        logger.info("Pipeline started (sha256=%s)", src_sha)
        with open(in_csv, "r", encoding="utf-8", newline="") as in_f:
            for row_num, row in enumerate(csv.DictReader(in_f), start=1):
                raw = row["hb_g_dl"].strip()
                if raw in ("", "N/A", "999"):
                    rejected += 1
                    logger.warning("Row %d rejected: %r", row_num, raw)
                else:
                    valid_vals.append(float(raw))

        mean_hb = (
            round(sum(valid_vals) / len(valid_vals), 2)
            if valid_vals
            else None
        )
        summary = {
            "source_sha256": src_sha,
            "valid_count": len(valid_vals),
            "rejected_count": rejected,
            "mean_hb": mean_hb,
        }
        logger.info(
            "Pipeline finished: valid=%d, rejected=%d",
            len(valid_vals),
            rejected,
        )
    finally:
        logger.removeHandler(handler)
        handler.close()

    with open(out_json, "w", encoding="utf-8") as out_f:
        json.dump(summary, out_f, indent=2)
    return summary
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch15_exam_01": audit_repo_hygiene,
    "ch15_exam_02": verify_data_provenance,
    "ch15_exam_03": run_logged_cleaning_pipeline,
}

tutor.check_exam("ch15", exam_submissions)


---
## Part III Capstone Debugging Post-Mortem (Solution Notebook Only)

- **Task 15.1 (Exact File vs. Directory Checks):** Using `.is_file()` for `README.md`, `.gitignore`, and `.env` and `.is_dir()` for `.ipynb_checkpoints` ensures strict boolean returns and guards against misclassified paths.
- **Task 15.2 (Checking Existence Before Hashing):** Calling `fpath.read_bytes()` before checking `if not fpath.is_file():` crashes with `FileNotFoundError` whenever a dataset is missing instead of recording it cleanly under `"missing"`.
- **Task 15.3 (Handler Teardown & Empty-Cohort Guard):** Always remove and close `logging.FileHandler` in `finally:` so temporary folders clean up cleanly and subsequent runs never duplicate log lines; guard `mean_hb` with `if valid_vals else None` so all-sentinel cohorts serialize cleanly as `null` instead of crashing with `ZeroDivisionError`.